In [84]:
import scanpy as sc
import pandas as pd
import numpy as np
import argparse
import scipy.sparse as sp
import os
import glob

In [76]:
raw = sc.read_h5ad("D:/tabula/raw/blood.h5ad")

In [78]:
raw

AnnData object with n_obs × n_vars = 85233 × 61806
    obs: 'donor', 'tissue', 'anatomical_position', 'method', 'cdna_plate', 'library_plate', 'notes', 'cdna_well', 'old_index', 'assay', 'sample_id', 'replicate', '10X_run', '10X_barcode', 'ambient_removal', 'donor_method', 'donor_assay', 'donor_tissue', 'donor_tissue_assay', 'cell_ontology_class', 'cell_ontology_id', 'compartment', 'broad_cell_class', 'free_annotation', 'manually_annotated', 'published_2022', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'total_counts_ercc', 'pct_counts_ercc', '_scvi_batch', '_scvi_labels', 'scvi_leiden_donorassay_full', 'age', 'sex', 'ethnicity', 'scvi_leiden_res05_tissue', 'sample_number'
    var: 'ensembl_id', 'gene_symbol', 'genome', 'mt', 'ercc', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'mean', 'std'
    uns: '_scvi_manager_uuid', '_scvi_uuid', '_training_mode', 'age_colors', 'assay_colors', 'compartment_colors', 'donor_colors', 'leide

In [80]:
layer = "log_normalized"
X = raw.layers[layer]

In [82]:
X

<85233x61806 sparse matrix of type '<class 'numpy.float32'>'
	with 198414481 stored elements in Compressed Sparse Row format>

In [50]:
if sp.issparse(X):
    X = X.tocsc()  # column-oriented for fast X.T @ G

In [51]:
group_col = "cell_ontology_class"  
groups = raw.obs[group_col].astype(str).values
cats, inv = np.unique(groups, return_inverse=True)  # cats = group names, inv = group index per cell
n_groups = len(cats)

In [54]:
raw.obs["tissue"]

TSP2_Bladder_NA_SS2_B114660_B111652_Stromal_J15    Bladder
TSP2_Bladder_NA_SS2_B114660_B111652_Stromal_O22    Bladder
TSP2_Bladder_NA_SS2_B114660_B111652_Stromal_A2     Bladder
TSP2_Bladder_NA_SS2_B114660_B111652_Stromal_H3     Bladder
TSP2_Bladder_NA_SS2_B114660_B111652_Stromal_L16    Bladder
                                                    ...   
TSP2_Bladder_NA_10X_1_2_TTTGTTGGTGACTATC           Bladder
TSP2_Bladder_NA_10X_1_2_TTTGTTGGTGAGGCAT           Bladder
TSP2_Bladder_NA_10X_1_2_TTTGTTGGTGGGCTCT           Bladder
TSP2_Bladder_NA_10X_1_2_TTTGTTGGTTTCGTTT           Bladder
TSP2_Bladder_NA_10X_1_2_TTTGTTGTCTAGCATG           Bladder
Name: tissue, Length: 66385, dtype: category
Categories (1, object): ['Bladder']

In [56]:
G = sp.csr_matrix((np.ones(raw.n_obs, dtype=np.float32),
                   (np.arange(raw.n_obs), inv)),
                  shape=(raw.n_obs, n_groups))

In [58]:
G

<66385x22 sparse matrix of type '<class 'numpy.float32'>'
	with 66385 stored elements in Compressed Sparse Row format>

In [62]:
sum_by_group = X.T @ G                                  # (n_vars × n_groups)
counts = np.asarray(G.sum(axis=0)).ravel().astype(float)
means = sum_by_group / counts                           # broadcasting division
is_expr = (X > 0.0) if sp.issparse(X) else (X > 0.0)
pct_expr = (is_expr.T @ G) / counts

# 6) Row (gene) and column (group) labels
gene_id = (raw.var["ensembl_id"].astype(str).values
           if "ensembl_id" in raw.var.columns else raw.var_names.astype(str).values)

gene_symbol = (raw.var["gene_symbol"].astype(str).values
               if "gene_symbol" in raw.var.columns else gene_id)

# sanity
print(len(gene_id), len(set(gene_id)))

61806 61806


In [30]:
def to_df(M):
    if sp.issparse(M): M = M.toarray()
    df = pd.DataFrame(M, index=gene_id, columns=cats)
    return df

In [64]:
df_mean = to_df(means)
df_pct  = to_df(pct_expr)

In [66]:
df_mean

,b cell,bladder urothelial cell,capillary endothelial cell,"cd4-positive, alpha-beta t cell","cd8-positive, alpha-beta t cell",endothelial cell,endothelial cell of lymphatic vessel,erythrocyte,fibroblast,macrophage,...,monocyte,myofibroblast cell,natural killer cell,neutrophil,pericyte,plasma cell,regulatory t cell,smooth muscle cell,t cell,vein endothelial cell
ENSG00000000003.15,0.000000,0.463651,0.087265,0.001364,0.000095,0.175552,0.204046,0.010462,0.093053,0.001986,...,0.000747,0.087060,0.000000,0.004219,0.021274,0.001880,0.000000,0.022573,0.001542,0.122728
ENSG00000000005.6,0.000000,0.000251,0.001832,0.000000,0.000000,0.003444,0.000000,0.000000,0.009217,0.000000,...,0.000069,0.000839,0.000000,0.000000,0.001845,0.001314,0.000000,0.000000,0.000000,0.006219
ENSG00000000419.14,0.284051,0.344316,0.341207,0.318147,0.317886,0.300692,0.331317,0.018812,0.274236,0.249331,...,0.242105,0.289266,0.297074,0.135743,0.234862,0.202383,0.322374,0.173032,0.311461,0.318820
ENSG00000000457.14,0.083133,0.064226,0.054826,0.116795,0.096365,0.017474,0.054638,0.000000,0.045654,0.046571,...,0.040197,0.045072,0.159278,0.021622,0.043421,0.058480,0.200717,0.063746,0.094685,0.060781
ENSG00000000460.17,0.089207,0.025043,0.068033,0.136440,0.051831,0.072124,0.021098,0.000000,0.019188,0.057050,...,0.028893,0.026639,0.242863,0.003967,0.022754,0.019173,0.479078,0.024741,0.064011,0.130842
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ENSG00000290162.1,0.000815,0.000077,0.000000,0.000401,0.000000,0.000000,0.000000,0.000000,0.000088,0.000315,...,0.000000,0.000119,0.000000,0.000000,0.000000,0.000000,0.000000,0.000795,0.000000,0.000000
ENSG00000290163.1,0.000000,0.000201,0.000000,0.000000,0.000265,0.000000,0.000000,0.000000,0.000525,0.001375,...,0.000212,0.000241,0.004160,0.000000,0.000000,0.000000,0.011216,0.001591,0.000000,0.000000
ENSG00000290164.1,0.000000,0.000147,0.000000,0.000106,0.000000,0.000000,0.000000,0.000000,0.000143,0.000357,...,0.000078,0.000000,0.000000,0.000000,0.000467,0.000000,0.000000,0.000000,0.000000,0.000000
ENSG00000290165.1,0.001785,0.001568,0.000000,0.001490,0.000642,0.000000,0.007207,0.000000,0.001002,0.002135,...,0.002214,0.000555,0.000000,0.002828,0.000413,0.000439,0.000000,0.001463,0.001418,0.000442


In [68]:
df_pct

,b cell,bladder urothelial cell,capillary endothelial cell,"cd4-positive, alpha-beta t cell","cd8-positive, alpha-beta t cell",endothelial cell,endothelial cell of lymphatic vessel,erythrocyte,fibroblast,macrophage,...,monocyte,myofibroblast cell,natural killer cell,neutrophil,pericyte,plasma cell,regulatory t cell,smooth muscle cell,t cell,vein endothelial cell
ENSG00000000003.15,0.000000,0.685034,0.177570,0.000914,0.000226,0.378378,0.228346,0.009434,0.160032,0.002538,...,0.001854,0.130102,0.000000,0.004149,0.025338,0.005809,0.000000,0.036087,0.001280,0.194064
ENSG00000000005.6,0.000000,0.000991,0.009346,0.000000,0.000000,0.027027,0.000000,0.000000,0.015400,0.000000,...,0.000824,0.001594,0.000000,0.000000,0.001689,0.001660,0.000000,0.000000,0.000000,0.011416
ENSG00000000419.14,0.278926,0.630153,0.551402,0.333942,0.263538,0.486486,0.401575,0.009434,0.386968,0.305838,...,0.360247,0.364477,0.303030,0.103734,0.315034,0.365975,0.521739,0.248813,0.300896,0.429224
ENSG00000000457.14,0.107438,0.170405,0.121495,0.146185,0.088673,0.135135,0.086614,0.000000,0.073878,0.058376,...,0.075386,0.063776,0.181818,0.016598,0.065878,0.129461,0.304348,0.094017,0.110115,0.109589
ENSG00000000460.17,0.105372,0.072978,0.121495,0.148013,0.047383,0.135135,0.031496,0.000000,0.031600,0.076777,...,0.054377,0.035077,0.242424,0.008299,0.027872,0.041494,0.608696,0.029440,0.071703,0.196347
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ENSG00000290162.1,0.002066,0.000367,0.000000,0.000457,0.000000,0.000000,0.000000,0.000000,0.000145,0.000635,...,0.000000,0.000319,0.000000,0.000000,0.000000,0.000000,0.000000,0.000950,0.000000,0.000000
ENSG00000290163.1,0.000000,0.000844,0.000000,0.000000,0.000226,0.000000,0.000000,0.000000,0.001090,0.001904,...,0.000412,0.000319,0.006061,0.000000,0.000000,0.000000,0.043478,0.001899,0.000000,0.000000
ENSG00000290164.1,0.000000,0.000404,0.000000,0.000914,0.000000,0.000000,0.000000,0.000000,0.000291,0.000317,...,0.000206,0.000000,0.000000,0.000000,0.000845,0.000000,0.000000,0.000000,0.000000,0.000000
ENSG00000290165.1,0.002066,0.006020,0.000000,0.002284,0.000903,0.000000,0.015748,0.000000,0.002252,0.004442,...,0.003708,0.000957,0.000000,0.004149,0.000845,0.001660,0.000000,0.002849,0.001280,0.002283


In [70]:
sym_map = pd.Series(gene_symbol, index=gene_id).groupby(level=0).first()

long = (
  df_mean.stack().rename("MeanLogNorm").to_frame()
    .join(df_pct.stack().rename("PctExpr"))
    .reset_index()
    .rename(columns={"level_0":"GeneID", "level_1":"CellType"})
)

long["GeneSymbol"] = long["GeneID"].map(sym_map)

# column order
long = long[["GeneID","GeneSymbol","CellType","MeanLogNorm","PctExpr"]]

In [72]:
long

,GeneID,GeneSymbol,CellType,MeanLogNorm,PctExpr
0,ENSG00000000003.15,TSPAN6,b cell,0.000000,0.000000
1,ENSG00000000003.15,TSPAN6,bladder urothelial cell,0.463651,0.685034
2,ENSG00000000003.15,TSPAN6,capillary endothelial cell,0.087265,0.177570
3,ENSG00000000003.15,TSPAN6,"cd4-positive, alpha-beta t cell",0.001364,0.000914
4,ENSG00000000003.15,TSPAN6,"cd8-positive, alpha-beta t cell",0.000095,0.000226
...,...,...,...,...,...
1359727,ENSG00000290166.1,ENSG00000290166,plasma cell,0.000000,0.000000
1359728,ENSG00000290166.1,ENSG00000290166,regulatory t cell,0.000000,0.000000
1359729,ENSG00000290166.1,ENSG00000290166,smooth muscle cell,0.000286,0.000950
1359730,ENSG00000290166.1,ENSG00000290166,t cell,0.000000,0.000000


In [74]:
long.to_csv("Tabula_bladder_long.csv", index=False)

# All files

In [87]:
in_dir  = r"D:/tabula/raw"          # folder with 28 .h5ad files
out_dir = r"D:/tabula/processed_long"    # where to save outputs
os.makedirs(out_dir, exist_ok=True)

def to_df(M):
    if sp.issparse(M): M = M.toarray()
    df = pd.DataFrame(M, index=gene_id, columns=cats)
    return df

def organ_from_path(p):
    return os.path.splitext(os.path.basename(p))[0]

In [95]:
for path in sorted(glob.glob(os.path.join(in_dir, "*.h5ad"))):
    organ = organ_from_path(path)
    print(f"\n=== Processing: {organ} ===")

    raw = sc.read_h5ad(path)

    layer = "log_normalized"
    X = raw.layers[layer]
    if sp.issparse(X):
        X = X.tocsc()  # column-oriented for fast X.T @ G
    
    group_col = "cell_ontology_class"  
    groups = raw.obs[group_col].astype(str).values
    cats, inv = np.unique(groups, return_inverse=True)  # cats = group names, inv = group index per cell
    n_groups = len(cats)

    G = sp.csr_matrix((np.ones(raw.n_obs, dtype=np.float32),
                   (np.arange(raw.n_obs), inv)),
                  shape=(raw.n_obs, n_groups))

    sum_by_group = X.T @ G                                  # (n_vars × n_groups)
    counts = np.asarray(G.sum(axis=0)).ravel().astype(float)
    means = sum_by_group / counts                           # broadcasting division
    is_expr = (X > 0.0) if sp.issparse(X) else (X > 0.0)
    pct_expr = (is_expr.T @ G) / counts

    # 6) Row (gene) and column (group) labels
    gene_id = (raw.var["ensembl_id"].astype(str).values
               if "ensembl_id" in raw.var.columns else raw.var_names.astype(str).values)

    gene_symbol = (raw.var["gene_symbol"].astype(str).values
               if "gene_symbol" in raw.var.columns else gene_id)

    # sanity
    print(len(gene_id), len(set(gene_id)))

    df_mean = to_df(means)
    df_pct  = to_df(pct_expr)

    sym_map = pd.Series(gene_symbol, index=gene_id).groupby(level=0).first()

    long = (
      df_mean.stack().rename("MeanLogNorm").to_frame()
        .join(df_pct.stack().rename("PctExpr"))
        .reset_index()
        .rename(columns={"level_0":"GeneID", "level_1":"CellType"})
    )

    long["GeneSymbol"] = long["GeneID"].map(sym_map)

    # column order
    long = long[["GeneID","GeneSymbol","CellType","MeanLogNorm","PctExpr"]]
    # --- save ---
    out_csv = os.path.join(out_dir, f"Tabula_{organ}_long.csv")
    long.to_csv(out_csv, index=False)
    print("Saved:", out_csv)


=== Processing: bladder ===
61806 61806
Saved: D:/tabula/processed_long\Tabula_bladder_long.csv

=== Processing: blood ===
61806 61806
Saved: D:/tabula/processed_long\Tabula_blood_long.csv

=== Processing: bone_marrow ===
61806 61806
Saved: D:/tabula/processed_long\Tabula_bone_marrow_long.csv

=== Processing: ear ===
61806 61806
Saved: D:/tabula/processed_long\Tabula_ear_long.csv

=== Processing: eye ===
61806 61806
Saved: D:/tabula/processed_long\Tabula_eye_long.csv

=== Processing: fat ===
61806 61806
Saved: D:/tabula/processed_long\Tabula_fat_long.csv

=== Processing: heart ===
61806 61806
Saved: D:/tabula/processed_long\Tabula_heart_long.csv

=== Processing: kidney ===
61806 61806
Saved: D:/tabula/processed_long\Tabula_kidney_long.csv

=== Processing: l.intestine ===
61806 61806
Saved: D:/tabula/processed_long\Tabula_l.intestine_long.csv

=== Processing: liver ===
61806 61806
Saved: D:/tabula/processed_long\Tabula_liver_long.csv

=== Processing: lung ===
61806 61806
Saved: D:/tabu